# 04 — Early warning & watchlist

**Plain English:** Monitoring is forward-looking, so we flag loans that are
*deteriorating right now*: at each loan's latest observed month, is it in Stage 2
(or worse) and/or did its delinquency get worse versus the prior month? Those
loans form a **watchlist** — the table a credit officer would actually work.

**One-line terms**
- **Deteriorating** — this month's bucket is worse than last month's.
- **Watchlist** — currently-active loans in Stage 2/3 or freshly deteriorating.
- Loan IDs are **masked** in the committed snapshot (no loan-level redistribution).

We only look at loans still active at the end of the sample (not already prepaid/defaulted-out).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")

In [ ]:
# Take each loan's latest observed month ---------------------------------
order = {"Current": 0, "30": 1, "60": 2, "90+": 3, "Default": 4}
panel["bucket_rank"] = panel["bucket"].map(order)
panel = panel.sort_values(["loan_seq", "mob"])
panel["prev_rank"] = panel.groupby("loan_seq")["bucket_rank"].shift(1)
latest = panel.groupby("loan_seq").tail(1).copy()

# Active = not closed out by a terminal zero-balance event
latest = latest[latest["zb_code"].isna()]
latest["deteriorating"] = latest["bucket_rank"] > latest["prev_rank"]
print(f"{len(latest):,} active loans at latest observation")

In [ ]:
# Watchlist: Stage 2/3 now, or deteriorating this month ------------------
watch = latest[(latest.stage.isin(["Stage 2", "Stage 3"])) | (latest.deteriorating)].copy()
watch["loan_id"] = m.mask_loan_id(watch["loan_seq"])
watch = watch.sort_values(["bucket_rank", "cur_upb"], ascending=False)
cols = ["loan_id", "vintage", "prop_state", "period", "bucket", "stage",
        "deteriorating", "loan_age", "cur_upb", "credit_score", "ltv"]
watch = watch[cols]
print(f"watchlist: {len(watch):,} loans  |  exposure on watch: ${watch.cur_upb.sum()/1e6:,.1f}m")
watch.head(15)

In [ ]:
# Summary of the watchlist by vintage + stage, and save -----------------
wsumm = (watch.groupby(["vintage", "stage"])
         .agg(loans=("loan_id", "size"), exposure_upb=("cur_upb", "sum"))
         .reset_index())
wsumm["exposure_upb"] = wsumm["exposure_upb"].round(0)
watch.to_csv(m.OUT_TABLES / "04_watchlist.csv", index=False)
wsumm.to_csv(m.OUT_TABLES / "04_watchlist_summary.csv", index=False)
print("saved watchlist + summary"); wsumm